# 第 18 章 · MoveIt 2：机械臂运动规划

本章回答：让机械臂"把手移到杯子上方 10 厘米、别撞桌子"，MoveIt 2 内部经过哪几步？SRDF 是什么？路径和轨迹有什么区别？规划完怎么交给第 16 章的轨迹控制器执行？实时遥操作 / 视觉伺服用什么？

配套讲义：仓库里 `ros2/notes/lessons/18-moveit2.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 MoveIt 2。这里用一个极简的时间参数化：每一段按"最慢的那个关节"排时间，只管速度上限、不管加速度，所以速度在路点处会跳变。MoveIt 推荐的 TOTG 同时满足速度和加速度上限，结果更平滑。

In [ ]:
import itertools

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 2 步 ④ 适配器 | 起点略超限位就截到限位 | `check_start_state_bounds([1.0005, -0.2], ...)` | [1.0, -0.2] |
| 第 2 步 ⑤ 路径 | 只有位形，没有时间 | `path` | 3 个路点 |
| 第 2 步 ⑤ 时间参数化 | 排上时间变成轨迹 | `time_parameterize(path, v_max)` | t = 0、0.5、2.1 秒 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**函数 `check_start_state_bounds`**：CheckStartStateBounds 适配器的意思：真机读数略超出限位，就把起点截到限位上。

In [ ]:
def check_start_state_bounds(q: list[float], lower: list[float], upper: list[float]) -> list[float]:
    """CheckStartStateBounds 适配器的意思：真机读数略超出限位，就把起点截到限位上。"""
    return [min(max(x, lo), hi) for x, lo, hi in zip(q, lower, upper)]

**函数 `time_parameterize`**：路径（一串关节位形，没有时间）→ 轨迹（每个点的 time_from_start 和各关节速度）。

In [ ]:
def time_parameterize(path: list[list[float]], v_max: list[float]) -> list[dict]:
    """路径（一串关节位形，没有时间）→ 轨迹（每个点的 time_from_start 和各关节速度）。"""
    points = [{"positions": path[0], "time_from_start": 0.0, "velocities": [0.0] * len(path[0])}]
    t = 0.0
    for a, b in itertools.pairwise(path):
        dt = max(abs(qb - qa) / v for qa, qb, v in zip(a, b, v_max))  # 最慢的关节决定这一段要多久
        t += dt
        vel = [round((qb - qa) / dt, 3) for qa, qb in zip(a, b)]
        points.append({"positions": b, "time_from_start": round(t, 3), "velocities": vel})
    points[-1]["velocities"] = [0.0] * len(path[0])  # 终点停下
    return points

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 ④ · 规划请求适配器：起点 1.0005 略超上限 1.0，先修正到限位")
print("  ", check_start_state_bounds([1.0005, -0.2], [-1.0, -2.0], [1.0, 2.0]))

In [ ]:
print("\n第二步 ⑤ · 规划器给的是路径：只有位形，没有时间")
path = [[0.0, 0.0], [0.5, 0.2], [0.8, 1.0]]  # 两个关节、三个路点，都是编的
v_max = [1.0, 0.5]  # 每个关节的最高速度 rad/s（编的）
print("  路径:", path)

In [ ]:
print("\n第二步 ⑤（续）· 时间参数化：每一段由最慢的关节决定要多久")
for p in time_parameterize(path, v_max):
    print(f"  t={p['time_from_start']:<4} 位置 {p['positions']}  速度 {p['velocities']}")

## 3. 代码里才看得到的

1. **这只是最简单的时间参数化**：只管速度上限，不管加速度，所以速度在路点处会突变。MoveIt 推荐的 TOTG（Time-Optimal Trajectory Generation：时间 / 最优 / 轨迹 / 生成）同时满足速度和加速度上限，还尽量快。
2. **第二段由关节 2 决定**：关节 1 走 0.3 只要 0.3 秒，关节 2 走 0.8 要 1.6 秒，两个关节必须同时到，所以这一段 1.6 秒。
3. **路径和速度上限都是编的**。

## 4. 自己改一改

- 把关节 2 的速度上限改成 1.0，看总时长；
- 在路径中间再插一个点，看 `time_from_start` 怎么变。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **三份配置**：URDF 长什么样，SRDF 怎么分组、哪些不查碰撞，YAML 管限位求解器规划器。
- **流水线**：请求 → 逆解 → 规划器（OMPL 采样 / Pilz 工业 / CHOMP 优化）→ 适配器 → 时间参数化（TOTG）→ 动作交给 JTC。
- **路径没时间，轨迹有时间**。
- **用法**：RViz 点点点，MoveItPy 写脚本，Servo 做实时，MTC 搭任务。